In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Deploy ADK Agent in AI Engine

This notebook provides a step-by-step to deploy an Agent Created using Agent Development Kit on Agent Engine (ReasonEngine on Vertex AI)

**Important**: This notebook consider that the Agent was built with ADK and the agent files are inside an agent folder and the dependencies in a file config.yaml

Folders structure (example): 
```
parent_folder/
    agent_folder/
        __init__.py
        agent.py
        config.yaml
        .env
    deploy_agent_engine.ipynb
```

config.yaml (example): 
```yaml
agent_name: 'agent_name'
agent_display_name: 'Agent Name'
agent_description: 'Useful agent to help users'

deploy:
  dependencies: ['google-cloud-aiplatform[agent_engines]', 'google-adk', 'cloudpickle', 'pydantic']
```


### Setup and Config

In [1]:
# Checking the google-adk and google-cloud-aiplatform versions
!pip freeze | grep google-adk
!pip freeze | grep google-cloud-aiplatform

google-adk==1.16.0
google-cloud-aiplatform==1.126.1


In [ ]:
# Authentication on gcloud (if necessary)
!gcloud auth application-default login

In [2]:
# Basic Libraries
import os 
import vertexai
import yaml

# AI Engine on Vertex AI 
from vertexai import agent_engines

# Library for AI Engine with ADK
from vertexai.preview import reasoning_engines

# Just to view JSON response formatted
import json
from IPython.display import display,Markdown,JSON

# To load envvars dict from .env file
from dotenv import dotenv_values

/Users/speca/Dev/google/adk_bq_agent/.venv/lib/python3.12/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


In [3]:
# Load Agent Config
AGENT_DIR = "adk_bq_agent"

# Load environment variables from .env file from agent Directory 
from dotenv import load_dotenv
env_file = f'./{AGENT_DIR}/.env'
load_dotenv(env_file)

# Load config from agents (Params and dependencies for deploy)
with open(f'./{AGENT_DIR}/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# For Vertex AI SDK 
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT")
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION")
BUCKET = os.environ.get("GOOGLE_CLOUD_BUCKET")

### Instantiate Agent from Directory

In [4]:
# Importing Agent Module from AGENT_DIR folder
import importlib
agent_module = importlib.import_module(f"{AGENT_DIR.replace('/','.')}.agent")

# Instantiate the Assistant as an ADK App 
adk_agent = reasoning_engines.AdkApp(
    agent=agent_module.root_agent,
    enable_tracing=True
)

Auth ID: adk-bq-oauth
BigQuery context loaded successfully from 'bigquery_context.txt'


### Running Agent Local (Optional)

In [5]:
# Run a simple query
for event in adk_agent.stream_query(
    user_id="user",
    message="Hi, how can you help me?",
):
    pass

# Formatted output
display(Markdown(f"```json\n{json.dumps(event, indent=2,ensure_ascii=False)}\n```"))

/Users/speca/Dev/google/adk_bq_agent/.venv/lib/python3.12/site-packages/vertexai/preview/reasoning_engines/templates/adk.py:798: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/Users/speca/Dev/google/adk_bq_agent/.venv/lib/python3.12/site-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


```json
{
  "content": {
    "parts": [
      {
        "text": "I can help you by querying a BigQuery database. Tell me what information you need from the database, and I will write and execute a SQL query to retrieve it for you. I can answer questions about users, courses, assignments, grades, and more.\n\nFor example, you can ask me:\n* \"How many students are enrolled in 'Quantum Gastronomy Fundamentals'?\"\n* \"What is the average grade for 'Tarefa 1 - Q_GASTRO'?\"\n* \"List all courses created last month.\"\n* \"Who are the top 5 students by overall grade?\"\n\nWhat would you like to know today?"
      }
    ],
    "role": "model"
  },
  "finish_reason": "STOP",
  "usage_metadata": {
    "candidates_token_count": 131,
    "candidates_tokens_details": [
      {
        "modality": "TEXT",
        "token_count": 131
      }
    ],
    "prompt_token_count": 9510,
    "prompt_tokens_details": [
      {
        "modality": "TEXT",
        "token_count": 9510
      }
    ],
    "total_token_count": 9641,
    "traffic_type": "ON_DEMAND"
  },
  "avg_logprobs": -0.18487499324420026,
  "invocation_id": "e-0ec5c8d2-f19e-478c-817c-d9798fc76f50",
  "author": "bigquery_agent",
  "actions": {
    "state_delta": {},
    "artifact_delta": {},
    "requested_auth_configs": {},
    "requested_tool_confirmations": {}
  },
  "id": "b109585a-a6c2-48d9-91b2-082394bbcdfc",
  "timestamp": 1764018483.899296
}
```

### Deploy on Agent Engine

In [6]:
# Instantiate Vertex AI
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=BUCKET,
)

In [7]:
## Retrieve all existent Agent Engine on your project
for agent in agent_engines.list():
    print(f"============================ \nAgent: {agent.display_name}\nResoruce Name: {agent.resource_name}\nCreated/updated at: {agent.update_time} \n\n" )

Agent: ADK OAuth Agent
Resoruce Name: projects/267339081837/locations/us-central1/reasoningEngines/1878366082473918464
Created/updated at: 2025-11-24 20:52:17.482440+00:00 


Agent: Agente Recursos Humanos
Resoruce Name: projects/267339081837/locations/us-central1/reasoningEngines/8886459683871653888
Created/updated at: 2025-11-13 20:31:33.996791+00:00 


Agent: ADK VAIS Search Agent
Resoruce Name: projects/267339081837/locations/us-central1/reasoningEngines/5370837224755560448
Created/updated at: 2025-11-11 18:59:13.814593+00:00 




In [8]:
# Read Requirements for Agent from config file
# Usually this ['google-cloud-aiplatform[agent_engines]', 'google-adk', 'cloudpickle'] plus the packages that agent needs
requirements = config['deploy']['dependencies']
requirements

['google-cloud-aiplatform[agent_engines,adk]',
 'google-adk==1.16.0',
 'cloudpickle',
 'pydantic',
 'python-dotenv',
 'pyyaml',
 'google-cloud-bigquery']

In [9]:
# Extra packages from agent folder (This is all .py files inside Agent Directory)
extra_packages = [AGENT_DIR]
extra_packages

['adk_bq_agent']

In [10]:
# Load Variables on env_vars dict to be used when creating the Agent
env_vars = dotenv_values(dotenv_path=env_file)

# Remove GCP variables (this variables already are defined in Agent Engine and are reserved)
keys_to_remove = [
    "GOOGLE_GENAI_USE_VERTEXAI",
    "GOOGLE_CLOUD_PROJECT",
    "GOOGLE_CLOUD_LOCATION",
    "GOOGLE_CLOUD_BUCKET"
]

for key in keys_to_remove:
    env_vars.pop(key, None)


In [ ]:
#UPDATE REASONING ENGINE WITH ALREADY EXIST

# resource_name="projects/267339081837/locations/us-central1/reasoningEngines/5654634370024079360"

# # Deploy the Agent on AI Engine (This takes a few minutes)
# remote_agent = agent_engines.update(
#     resource_name,
#     agent_engine = adk_agent,             # The Agent instantiated as ADK agent
#     requirements=requirements,            # Requirements file
#     extra_packages=extra_packages,        # Extra packages
#     display_name= config['agent_display_name'],    # Display name  
#     description= config['agent_description'],     # Description
#     env_vars=env_vars                     # Env Vars dict
# )

In [11]:
# Deploy the Agent on AI Engine (This takes a few minutes)
remote_agent = agent_engines.create(
    agent_engine = adk_agent,             # The Agent instantiated as ADK agent
    requirements=requirements,            # Requirements file
    extra_packages=extra_packages,        # Extra packages
    display_name=config['agent_display_name'],    # Display name  
    description=config['agent_description'],     # Description
    env_vars=env_vars                     # Env Vars dict
)

Identified the following requirements: {'google-cloud-aiplatform': '1.126.1', 'cloudpickle': '3.1.1', 'pydantic': '2.11.7'}


INFO:vertexai.agent_engines:Identified the following requirements: {'google-cloud-aiplatform': '1.126.1', 'cloudpickle': '3.1.1', 'pydantic': '2.11.7'}


The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'google-adk==1.16.0', 'cloudpickle', 'pydantic', 'python-dotenv', 'pyyaml', 'google-cloud-bigquery']


INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'google-adk==1.16.0', 'cloudpickle', 'pydantic', 'python-dotenv', 'pyyaml', 'google-cloud-bigquery']


Using bucket ge-speca-sandbox-adk-deploy


...

INFO:vertexai.agent_engines:AgentEngine created. Resource name: projects/267339081837/locations/us-central1/reasoningEngines/2632719020058476544


To use this AgentEngine in another session:


INFO:vertexai.agent_engines:To use this AgentEngine in another session:


agent_engine = vertexai.agent_engines.get('projects/267339081837/locations/us-central1/reasoningEngines/2632719020058476544')


INFO:vertexai.agent_engines:agent_engine = vertexai.agent_engines.get('projects/267339081837/locations/us-central1/reasoningEngines/2632719020058476544')


### Test Remote Agent on Agent Engine

In [12]:
## Retrieve all existent Agent Engine resource.names (Agents)
# To confirm new agent was deployed
for agent in agent_engines.list():
    print(f"============================ \nAgent: {agent.display_name}\nResoruce Name: {agent.resource_name}\nCreated/updated at: {agent.update_time} \n\n" )

Agent: ADK BQ Agent OAuth
Resoruce Name: projects/267339081837/locations/us-central1/reasoningEngines/2632719020058476544
Created/updated at: 2025-11-24 21:16:19.103085+00:00 


Agent: ADK OAuth Agent
Resoruce Name: projects/267339081837/locations/us-central1/reasoningEngines/1878366082473918464
Created/updated at: 2025-11-24 20:52:17.482440+00:00 


Agent: Agente Recursos Humanos
Resoruce Name: projects/267339081837/locations/us-central1/reasoningEngines/8886459683871653888
Created/updated at: 2025-11-13 20:31:33.996791+00:00 


Agent: ADK VAIS Search Agent
Resoruce Name: projects/267339081837/locations/us-central1/reasoningEngines/5370837224755560448
Created/updated at: 2025-11-11 18:59:13.814593+00:00 




In [13]:
# Confirm that "remote_agent" is pointing to your new agent
print(f"=================== Remote Agent ============================ \n\
 Name: {remote_agent.display_name}\n\
 Resoruce Name: {remote_agent.resource_name}\n\
 Created/updated at: {remote_agent.update_time} \n\n" )

=================== Remote Agent ============================ 
 Name: ADK BQ Agent OAuth
 Resoruce Name: projects/267339081837/locations/us-central1/reasoningEngines/2632719020058476544
 Created/updated at: 2025-11-24 21:16:19.103085+00:00 




In [14]:
# Run a simple query
for remote_event in remote_agent.stream_query(
    user_id="user_test_deploy",
    message="Hi, what you can do form me?",
):
    display(JSON(remote_event,expanded=True)) 

<IPython.core.display.JSON object>

In [15]:
# Formatted final output
display(Markdown(f"```json\n{json.dumps(remote_event, indent=2,ensure_ascii=False)}\n```"))

```json
{
  "content": {
    "parts": [
      {
        "text": "I can help you by querying a BigQuery database. I can write and execute SQL queries to retrieve information about courses, users, assignments, grades, and more, from a Moodle-like learning management system.\n\nJust tell me what information you're looking for! For example, you can ask me:\n\n*   \"How many students are enrolled in 'Quantum Gastronomy Fundamentals'?\"\n*   \"What are the average grades for assignments in each course?\"\n*   \"List all courses created last year.\"\n*   \"Which users have not completed any modules?\"\n*   \"Show me the details of 'Tarefa 1 - Q_GASTRO'.\"\n\nFeel free to ask any question you have about the data!"
      }
    ],
    "role": "model"
  },
  "finish_reason": "STOP",
  "usage_metadata": {
    "candidates_token_count": 150,
    "candidates_tokens_details": [
      {
        "modality": "TEXT",
        "token_count": 150
      }
    ],
    "prompt_token_count": 9511,
    "prompt_tokens_details": [
      {
        "modality": "TEXT",
        "token_count": 9511
      }
    ],
    "total_token_count": 9661,
    "traffic_type": "ON_DEMAND"
  },
  "avg_logprobs": -0.21686375935872396,
  "invocation_id": "e-86b9d24a-949a-4c4a-a208-1a56a0c37469",
  "author": "bigquery_agent",
  "actions": {
    "state_delta": {},
    "artifact_delta": {},
    "requested_auth_configs": {},
    "requested_tool_confirmations": {}
  },
  "id": "169bc884-3045-4ef5-b1f8-1e5dffd40443",
  "timestamp": 1764019004.27813
}
```